 **Read CSV file**


In [0]:
spark

In [0]:
flight_df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .option("mode", "failfast") \
    .load("/Volumes/workspace/default/my_volume/flight_data.csv")

flight_df.show(5)

+-----------------+-------------------+-----+
|              _c0|                _c1|  _c2|
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df_Header=spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "false") \
    .option("mode", "failfast") \
    .load("/Volumes/workspace/default/my_volume/flight_data.csv")
flight_df_Header.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df_Header.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: string (nullable = true)



In [0]:
flight_df_Header_schema=spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .option("mode", "failfast") \
    .load("/Volumes/workspace/default/my_volume/flight_data.csv")
flight_df_Header.show(5) 

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df_Header_schema.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



**Create Schema**

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType


In [0]:
my_schema=StructType([StructField("DEST_COUNTRY_NAME",StringType(),True),
                       StructField("ORIGIN_COUNTRY_NAME",StringType(),True),
                       StructField("count",IntegerType(),True)])


In [0]:
flight_df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .schema(my_schema) \
    .option("mode", "permisive") \
    .load("/Volumes/workspace/default/my_volume/flight_data.csv")

flight_df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| NULL|
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
%fs
ls /Volumes/workspace/default/my_volume/flight_data.csv

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_volume/flight_data.csv,flight_data.csv,7375,1775758095000


In [0]:
flight_df = spark.read.format("csv") \
    .option("header", "false") \
    .option("SkipRows", "1")\
    .option("inferSchema", "false") \
    .schema(my_schema) \
    .option("mode", "permisive") \
    .load("/Volumes/workspace/default/my_volume/flight_data.csv")

flight_df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


### **Handling Corrupted Data**


In [0]:
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "permisive") \
    .load("/Volumes/workspace/default/my_volume1/employee.csv")

employee_df.show()

+---+--------+---+------+------------+--------+
| id|    name|age|salary|     address| nominee|
+---+--------+---+------+------------+--------+
|  1|  Manish| 26| 75000|       bihar|nominee1|
|  2|  Nikita| 23|100000|uttarpradesh|nominee2|
|  3|  Pritam| 22|150000|   Bangalore|   India|
|  4|Prantosh| 17|200000|     Kolkata|   India|
|  5|  Vikash| 31|300000|        NULL|nominee5|
+---+--------+---+------+------------+--------+



**Code run in failfast mode (failed execution when corrupt record found)**

In [0]:
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "Failfast") \
    .load("/Volumes/workspace/default/my_volume1/employee.csv")

employee_df.show()

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-8526972878409314>, line 7
      1 employee_df = spark.read.format("csv") \
      2     .option("header", "true") \
      3     .option("inferSchema", "true") \
      4     .option("mode", "Failfast") \
      5     .load("/Volumes/workspace/default/my_volume1/employee.csv")
----> 7 employee_df.show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894    

## How can we print bad Record
### ANS:Create Schema

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

In [0]:
emp_schema=StructType([StructField("id",IntegerType(),True),
                       StructField("name",StringType(),True),
                       StructField("age",IntegerType(),True),
                       StructField("salary",IntegerType(),True),
                       StructField("address",StringType(),True),
                       StructField("nominee",StringType(),True),
                       StructField("_corrupted_record",StringType(),True)])


In [0]:
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(emp_schema) \
    .option("mode", "permissive") \
    .load("/Volumes/workspace/default/my_volume1/employee.csv")

employee_df.show(truncate=False)




+---+--------+---+------+------------+--------+-----------------+
|id |name    |age|salary|address     |nominee |_corrupted_record|
+---+--------+---+------+------------+--------+-----------------+
|1  |Manish  |26 |75000 |bihar       |nominee1|NULL             |
|2  |Nikita  |23 |100000|uttarpradesh|nominee2|NULL             |
|3  |Pritam  |22 |150000|Bangalore   |India   |nominee3         |
|4  |Prantosh|17 |200000|Kolkata     |India   |nominee4         |
|5  |Vikash  |31 |300000|NULL        |nominee5|NULL             |
+---+--------+---+------+------------+--------+-----------------+



## **How to we store corrupted records and how can we access it later**

In [0]:
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .schema(emp_schema) \
    .option("badRecordsPath","/Volumes/workspace/default/my_volume1/bad_records")\
    .load("/Volumes/workspace/default/my_volume1/employee.csv")
    

employee_df.show()

+---+--------+---+------+---------+-------+-----------------+
| id|    name|age|salary|  address|nominee|_corrupted_record|
+---+--------+---+------+---------+-------+-----------------+
|  3|  Pritam| 22|150000|Bangalore|  India|         nominee3|
|  4|Prantosh| 17|200000|  Kolkata|  India|         nominee4|
+---+--------+---+------+---------+-------+-----------------+



In [0]:
%fs
ls /Volumes/workspace/default/my_volume1

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_volume1/employee.csv,employee.csv,232,1776011124000


In [0]:
%fs
ls /Volumes/workspace/default/my_volume1/bad_records/20260412T185306/bad_records/

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_volume1/bad_records/20260412T185306/bad_records/part-00000-98357534-b50f-4d97-8469-8495731d8f4c,part-00000-98357534-b50f-4d97-8469-8495731d8f4c,553,1776019989000


### Above data in jSON format so i want to read and print so i need to cread new data frame Df

In [0]:
bad_data_df=spark.read.format("json").load ("/Volumes/workspace/default/my_volume1/bad_records/20260412T185306/bad_records/")
bad_data_df.show()

+--------------------+--------------------+--------------------+
|                path|              reason|              record|
+--------------------+--------------------+--------------------+
|dbfs:/Volumes/wor...|org.apache.spark....|1,Manish,26,75000...|
|dbfs:/Volumes/wor...|org.apache.spark....|2,Nikita,23,10000...|
|dbfs:/Volumes/wor...|org.apache.spark....|5,Vikash,31,30000...|
+--------------------+--------------------+--------------------+

